# A/B Testing Activation Calculation

This notebook reads raw `USERS_RAW` and `TRACKS_RAW` tables from Snowflake into pandas.

Then we do the A/B assignment and activation calculation in pandas.

Activation means the user completed both `workspace_created` and `report_generated` within 7 days of `first_logged_in_at`.

In [ ]:
from pathlib import Path
import os

import pandas as pd
import snowflake.connector



conn = snowflake.connector.connect(
    account='ep58496.us-east-2.aws',
    user='NAGHMEH6',
    private_key_file='/path/to/rsa_key.p8',  # path to your private key
    authenticator='SNOWFLAKE_JWT',            # THIS IS KEY
    warehouse='COMPUTE_WH',
    role='ACCOUNTADMIN'
)
def load_env(path: str = ".env") -> None:
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_env()

In [8]:


tracks.event_name.value_counts()

event_name
api_call_made            21466
file_downloaded          15124
notification_sent         9584
workspace_created         8548
report_generated          7377
login_successful          7341
workflow_triggered        6918
message_sent              5981
report_exported           4861
dashboard_viewed          2075
file_uploaded             1286
workflow_created          1264
mention_used              1138
ticket_created            1095
password_reset             925
role_assigned              822
help_center_visited        732
ticket_resolved            689
custom_field_added         503
login_failed               366
ticket_reopened            355
integration_failed         354
integration_connected      288
workspace_deleted          283
feature_enabled            204
branding_updated           152
feature_disabled           142
2fa_enabled                 81
audit_log_viewed            53
Name: count, dtype: int64

In [5]:
import hashlib
import os
private_key_file='/path/to/rsa_key.p8'
connection_params = {
    "private_key_file": "/Users/naghmeh6/agentic_ai/ABtesting/rsa_key.p8",
    "authenticator": os.environ["SNOWFLAKE_AUTHENTICATOR"],
    "account": os.environ["SNOWFLAKE_ACCOUNT"],
    "user": os.environ["SNOWFLAKE_USERNAME"],
    "password": os.environ["SNOWFLAKE_PASSWORD"],
    "warehouse": os.environ["SNOWFLAKE_WAREHOUSE"],
    "database": os.environ["SNOWFLAKE_DATABASE"],
    "schema": os.environ["SNOWFLAKE_SCHEMA"],
}

if os.getenv("SNOWFLAKE_ROLE"):
    connection_params["role"] = os.environ["SNOWFLAKE_ROLE"]

if os.getenv("SNOWFLAKE_AUTHENTICATOR"):
    connection_params["authenticator"] = os.environ["SNOWFLAKE_AUTHENTICATOR"]

with snowflake.connector.connect(**connection_params) as conn:
    users = pd.read_sql("SELECT * FROM USERS_RAW", conn)
    tracks = pd.read_sql("SELECT * FROM TRACKS_RAW", conn)

users.columns = users.columns.str.lower()
tracks.columns = tracks.columns.str.lower()

users["first_logged_in_at"] = pd.to_datetime(users["first_logged_in_at"])
tracks["event_timestamp"] = pd.to_datetime(tracks["event_timestamp"])

def assign_variant(user_id: str) -> str:
    hash_value = int(hashlib.md5(user_id.encode()).hexdigest(), 16)
    return "treatment" if hash_value % 2 else "control"

assignments = users[["user_id"]].copy()
assignments["variant"] = assignments["user_id"].apply(assign_variant)

activation_events = tracks[tracks["event_name"].isin(["workspace_created", "report_generated"])]
activation_events = activation_events.merge(
    users[["user_id", "first_logged_in_at"]],
    on="user_id",
    how="inner",
)

activation_events = activation_events[
    (activation_events["event_timestamp"] >= activation_events["first_logged_in_at"])
    & (activation_events["event_timestamp"] <= activation_events["first_logged_in_at"] + pd.Timedelta(days=7))
]

user_activation = (
    activation_events
    .pivot_table(
        index="user_id",
        columns="event_name",
        values="event_timestamp",
        aggfunc="min",
    )
    .reset_index()
)

user_activation["is_activated"] = (
    user_activation["workspace_created"].notna()
    & user_activation["report_generated"].notna()
)

experiment_users = users[["user_id", "first_logged_in_at"]].merge(assignments, on="user_id")
experiment_users = experiment_users.merge(
    user_activation[["user_id", "is_activated"]],
    on="user_id",
    how="left",
)
experiment_users["is_activated"] = experiment_users["is_activated"].fillna(False).astype(bool)

summary = (
    experiment_users
    .groupby("variant", as_index=False)
    .agg(
        users=("user_id", "count"),
        activated_users=("is_activated", "sum"),
    )
)
summary["activation_rate"] = summary["activated_users"] / summary["users"]
summary

/var/folders/35/mcj68sf96mz81d075p4nxy3m0000gp/T/ipykernel_68120/495204383.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  users = pd.read_sql("SELECT * FROM USERS_RAW", conn)
/var/folders/35/mcj68sf96mz81d075p4nxy3m0000gp/T/ipykernel_68120/495204383.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tracks = pd.read_sql("SELECT * FROM TRACKS_RAW", conn)
/var/folders/35/mcj68sf96mz81d075p4nxy3m0000gp/T/ipykernel_68120/495204383.py:72: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_sile

,variant,users,activated_users,activation_rate
0,control,2977,78,0.026201
1,treatment,3029,80,0.026411


In [11]:
summary

,variant,users,activated_users,activation_rate
0,control,2977,78,0.026201
1,treatment,3029,80,0.026411


In [23]:
seeds_path = Path("/Users/naghmeh6/agentic_ai/ABtesting/lightdash-demo-saas/seeds")
i=0
for csv_file in sorted(seeds_path.glob("*.csv")):
    i=i+1
    if i>8:
        print(f"\n{csv_file.name}")
        print(pd.read_csv(csv_file).head(2).to_json())


product_purchases_raw.csv
{"customer_id":{"0":1,"1":2},"product_name":{"0":"Product A","1":"Product A"}}

product_region_customers_raw.csv
{"customer_id":{"0":1,"1":2},"customer_name":{"0":"Alice","1":"Bob"},"region":{"0":"North","1":"North"}}

sales_targets_raw.csv
{"target_type":{"0":"industry","1":"industry"},"target_value":{"0":"Financial Services","1":"Financial Services"},"quarter_start_date":{"0":"2023-01-01","1":"2023-04-01"},"target_deals":{"0":15,"1":18},"target_amount":{"0":37500,"1":45000}}

spain_regional_metrics_raw.csv
{"region_code":{"0":"ES-MD","1":"ES-CT"},"region_name":{"0":"Community of Madrid","1":"Catalonia"},"capital_city":{"0":"Madrid","1":"Barcelona"},"latitude":{"0":40.4168,"1":41.3851},"longitude":{"0":-3.7038,"1":2.1734},"customer_count":{"0":185,"1":142},"annual_revenue_eur":{"0":1480000,"1":1065000},"employee_count":{"0":28,"1":22},"market_penetration_pct":{"0":8.2,"1":6.5},"yoy_growth_pct":{"0":22.5,"1":28.4}}

tracks_raw.csv
{"user_id":{"0":"d1b14251-fc

In [20]:
print(pd.read_csv('/Users/naghmeh6/agentic_ai/ABtesting/lightdash-demo-saas/seeds/addresses_raw.csv').head(2).to_json())

{"address_id":{"0":"f6bf993b-4318-4734-842e-115ca3a7b8ee","1":"50f09906-2ba3-4599-9323-ff8d1e9a3513"},"user_id":{"0":"e1e6cd80-18c9-41ab-bdef-1863593e373f","1":"683ece6f-329f-4ece-977d-c0c0ca84068e"},"street_address":{"0":"4878 Park Avenue","1":"4598 Madison Drive"},"city":{"0":"Melbourne","1":"Melbourne"},"state":{"0":"VIC","1":"VIC"},"postal_code":{"0":"2414","1":"3518"},"country_iso_code":{"0":"AUS","1":"AUS"},"valid_from":{"0":"2024-01-29 00:00:00","1":"2024-04-13 00:00:00"},"valid_to":{"0":null,"1":null}}


In [ ]:
addresses_raw.csv:
{"address_id":{"0":"f6bf993b-4318-4734-842e-115ca3a7b8ee","1":"50f09906-2ba3-4599-9323-ff8d1e9a3513"},"user_id":{"0":"e1e6cd80-18c9-41ab-bdef-1863593e373f","1":"683ece6f-329f-4ece-977d-c0c0ca84068e"},"street_address":{"0":"4878 Park Avenue","1":"4598 Madison Drive"},"city":{"0":"Melbourne","1":"Melbourne"},"state":{"0":"VIC","1":"VIC"},"postal_code":{"0":"2414","1":"3518"},"country_iso_code":{"0":"AUS","1":"AUS"},"valid_from":{"0":"2024-01-29 00:00:00","1":"2024-04-13 00:00:00"},"valid_to":{"0":null,"1":null}}



SyntaxError: invalid syntax (911342276.py, line 1)